# Create MacArthur Fellows Awards (FELLOWSHIP PATTERN, Crownpeak Solr search)

Ingest of the **MacArthur Fellowship** (popularly the "Genius Grant"), given annually since 1981 by the **John D. and Catherine T. MacArthur Foundation**. The fellowship is awarded to "talented individuals who have shown extraordinary originality and dedication in their creative pursuits and a marked capacity for self-direction" — there is no application; recipients are nominated anonymously. Each fellowship comes with an award of **$800,000** paid quarterly over 5 years.

**Awarding body:** John D. and Catherine T. MacArthur Foundation — `F4320306142` (US, DOI `10.13039/100000870`).

**Corpus** (verified 2026-05-22 full local fetch): **1,174 fellow rows** spanning **1981-2025** (45 distinct years). Notable: Stephen Jay Gould (1981), Cormac McCarthy (1981), Richard Stallman (1990), David Foster Wallace (1997), Ta-Nehisi Coates (2015), Lin-Manuel Miranda (2015), Rebecca Richards-Kortum (2016), Paul Rothemund (2007). Spans all fields from poetry to nanotechnology.

**Source: Crownpeak Solr search index** — first ingest on the project to use this CMS family. Distinct from prior methods:
- Drupal JSON:API (Lemelson #134)
- GraphQL via Nuxt SPA (Welch #133)
- Webflow CMS (WFP #131)
- WordPress REST (Hertz #125, CIFAR #110)
- FacetWP (Hewlett #121)

**Discovery method** (documented in script header for future contributors who encounter MacFound / similar Crownpeak-backed sites):
1. Loaded `/programs/awards/fellows/results` — page is a SPA with `/js/fellows.js` + `/js/crownpeak.searchg2-1.0.3.js`.
2. `fellows.js` contains the literal config: `collection: 'live-macfound-redesign-rt'`.
3. The searchg2 URL pattern in the JS bundle is `https://searchg2.crownpeak.net/%%COLLECTION%%/%%HANDLER%%`.
4. Hitting `searchg2.crownpeak.net/live-macfound-redesign-rt/select` returns a Solr response — public endpoint, no auth.
5. Filter `q=custom_s_template:"fellow page"` → 1,174 fellow docs.
6. Per-doc fields: `custom_s_name`, `custom_s_class_year`, `custom_s_title` (field-of-work descriptor), `custom_s_association` (home institution), `custom_s_country`, `custom_s_short_bio`, `custom_s_url`, `custom_s_age`, `custom_s_area_display` (primary area), and a few dozen other custom fields. Single Solr collection covers the entire foundation site (~71K docs total including grants + fellows + news + other content types).

**Coverage** (verified 2026-05-22 full extraction):
- 100% `funder_award_id`, `year`, `name`, `slug`, `field_title`, `affiliation`, `country`, `description`
- 1981-2025 / 45 distinct years

**Schema:**
- `funder_award_id` = `macarthur-fellow-{year}-{slug}` (e.g., `macarthur-fellow-2007-paul-rothemund`). Slug from `custom_s_url` (foundation-stable). Verified unique across 1,174 rows.
- `lead_investigator` PERSON-LEVEL: `given_name` + `family_name` from `split_name()` (runbook §2.4.1). `affiliation.name` = `custom_s_association` (the institutional affiliation at time of award, 100% populated); `affiliation.country` = `custom_s_country` (100% populated).
- `funder_scheme = 'MacArthur Fellowship'`. Single program.
- `funding_type = 'fellowship'`.
- `start_date` = `{year}-01-01`, `end_date` = `{year+4}-12-31` (5-year payout window per the foundation's documented payout structure).
- `declined` always FALSE.

**Amount choice — constant USD 800,000 per fellow:**
The foundation's `/programs/awards/fellows/about` page states verbatim: **"Each fellowship comes with an award of $800,000 to the recipient, paid out in equal quarterly installments over five years."** (Verified 2026-05-22.) The award was originally tied to age (1981 grants were $24K–$60K depending on age) and has been increased multiple times since (most recently to $800K in 2013). The foundation publishes only the current rule, so we ship $800K uniformly and document the assumption. §6.7 amount-coverage NOT waived.

**Source authority** — `macfound.org` is the foundation's own site. Method #2 (REST/search API) on the runbook ladder via Crownpeak Solr.

**Prerequisites**: Run `scripts/local/macarthur_fellows_to_s3.py` first to make ~3 paginated Solr calls (500 rows each, ~10 sec wall-clock total), write parquet, upload to S3.

**Data source**: `https://searchg2.crownpeak.net/live-macfound-redesign-rt/select?q=custom_s_template:"fellow page"`

**S3 location**: `s3a://openalex-ingest/awards/macarthur_fellows/macarthur_fellows.parquet`

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.macarthur_fellows_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/macarthur_fellows/macarthur_fellows.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.macarthur_fellows_raw;

## Step 1.5: Inspect raw + money/currency scan + coverage acknowledgment

All source columns from the Crownpeak Solr extraction. **Verified per-row coverage** (1,174 fellows, validated 2026-05-22):
- 100% `funder_award_id`, `year`, `name`, `slug`, `field_title`, `affiliation`, `country`, `description`
- 100% `amount` = constant USD 800,000 per fellow (foundation's documented current rule)
- 45 distinct years (1981-2025)

Source columns: `funder_award_id`, `year`, `slug`, `name`, `given_name`, `family_name`, `field_title`, `area`, `affiliation`, `country`, `age_at_award`, `display_name`, `description`, `amount`, `currency`, `start_date`, `end_date`, `landing_page_url`, `declined`.

In [ ]:
%sql
DESCRIBE openalex.awards.macarthur_fellows_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.macarthur_fellows_raw LIMIT 5;

In [ ]:
%sql
-- §1.5 money-shape scan — confirms constant USD 800K per fellow.
SELECT
    MIN(TRY_CAST(amount AS DOUBLE)) AS min_amount,
    MAX(TRY_CAST(amount AS DOUBLE)) AS max_amount,
    AVG(TRY_CAST(amount AS DOUBLE)) AS avg_amount,
    COUNT(amount) AS non_null,
    COUNT(*) AS total_rows
FROM openalex.awards.macarthur_fellows_raw;

In [ ]:
%sql
-- Year coverage sanity — should span 1981-2025 with ~20-30 fellows/year.
SELECT MIN(TRY_CAST(year AS INT)) AS min_yr,
       MAX(TRY_CAST(year AS INT)) AS max_yr,
       COUNT(DISTINCT year) AS distinct_years,
       COUNT(*) AS total_fellows
FROM openalex.awards.macarthur_fellows_raw;

In [ ]:
%sql
-- Field-of-work distribution (top 15) — confirms the broad span
-- (Poet, Novelist, Mathematician, Physicist, MacArthur fellows span
-- every field).
SELECT area,
       COUNT(*) AS fellows
FROM openalex.awards.macarthur_fellows_raw
GROUP BY area
ORDER BY fellows DESC
LIMIT 15;

## Step 1.6: Fail-fast — verify MacArthur Foundation funder row exists

In [ ]:
%sql
-- Must return exactly 1 row.
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320306142;

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.macarthur_fellows_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306142  -- John D. and Catherine T. MacArthur Foundation
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    n.display_name,
    n.description,
    f.funder_id,
    n.funder_award_id,
    TRY_CAST(n.amount AS DOUBLE) AS amount,
    n.currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'fellowship' AS funding_type,
    'MacArthur Fellowship' AS funder_scheme,
    'macarthur_fellows' AS provenance,
    TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS start_date,
    TRY_TO_DATE(n.end_date,   'yyyy-MM-dd') AS end_date,
    TRY_CAST(SUBSTRING(n.start_date, 1, 4) AS INT) AS start_year,
    TRY_CAST(SUBSTRING(n.end_date,   1, 4) AS INT) AS end_year,
    -- lead_investigator: PERSON-LEVEL. affiliation.name = institutional
    -- affiliation at time of award (100% populated from Solr custom_s_association),
    -- affiliation.country = self-reported country (100% populated).
    CASE
        WHEN n.name IS NULL OR n.name = '' THEN NULL
        ELSE struct(
            n.given_name AS given_name,
            n.family_name AS family_name,
            CAST(NULL AS STRING) AS orcid,
            TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS role_start,
            struct(
                n.affiliation AS name,
                n.country AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.landing_page_url AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    n.declined,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.macarthur_fellows_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL
  AND n.name IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 104

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'macarthur_fellows' AND priority = 104;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    104 AS priority  -- MacArthur Fellows priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.macarthur_fellows_awards;

## Step 6: Verification

Full §6.1-6.7. Amount-coverage NOT waived (constant $800K per fellow). Verified expectations from the 2026-05-22 extraction:
- Row count: **1,174**
- 100% `pct_amount` (constant $800,000 USD per fellow)
- `currencies = [USD]`
- 1 distinct `funder_scheme` ('MacArthur Fellowship'), 1 distinct `funding_type` ('fellowship')
- 45 distinct years (1981-2025)
- 100% affiliation + country populated
- `declined = FALSE` everywhere

In [ ]:
%sql
SELECT COUNT(*) AS total_macarthur_award_rows FROM openalex.awards.macarthur_fellows_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.macarthur_fellows_awards;

In [ ]:
%sql
-- §6.3 Data completeness — expect 100% across all fields.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_pi,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(start_date), COUNT(*)) * 100.0, 1) AS pct_dates,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description
FROM openalex.awards.macarthur_fellows_awards;

In [ ]:
%sql
-- §6.7 amount + currency fail-fast (NOT waived).
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(MAX(amount), 0) AS max_amount,
    ROUND(AVG(amount), 0) AS avg_amount
FROM openalex.awards.macarthur_fellows_awards;

In [ ]:
%sql
-- Year distribution + total fellowship dollars
SELECT start_year,
       COUNT(*) AS rows,
       ROUND(SUM(amount)/1e6, 2) AS total_usd_millions
FROM openalex.awards.macarthur_fellows_awards
WHERE start_year IS NOT NULL
GROUP BY start_year
ORDER BY start_year DESC
LIMIT 100;

In [ ]:
%sql
-- Sample notable recent + earliest fellows
SELECT id, SUBSTRING(display_name, 1, 80) AS title,
       lead_investigator.given_name AS given,
       lead_investigator.family_name AS family,
       SUBSTRING(lead_investigator.affiliation.name, 1, 50) AS affiliation,
       lead_investigator.affiliation.country AS country,
       start_year
FROM openalex.awards.macarthur_fellows_awards
WHERE start_year >= 2024 OR start_year <= 1982
ORDER BY start_year DESC
LIMIT 15;

In [ ]:
%sql
-- Top 10 institutions by MacArthur Fellow count
SELECT lead_investigator.affiliation.name AS affiliation,
       COUNT(*) AS fellows
FROM openalex.awards.macarthur_fellows_awards
WHERE lead_investigator.affiliation.name IS NOT NULL
GROUP BY affiliation
ORDER BY fellows DESC
LIMIT 10;

In [ ]:
%sql
-- Country distribution
SELECT lead_investigator.affiliation.country AS country,
       COUNT(*) AS fellows
FROM openalex.awards.macarthur_fellows_awards
WHERE lead_investigator.affiliation.country IS NOT NULL
GROUP BY country
ORDER BY fellows DESC
LIMIT 10;

In [ ]:
%sql
-- Funder struct populated correctly?
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.macarthur_fellows_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;

In [ ]:
%sql
-- Declined must be FALSE everywhere
SELECT declined, COUNT(*) AS rows
FROM openalex.awards.macarthur_fellows_awards
GROUP BY declined;